# VAE JAX CelebA Kaggle TPU v5e-8 Pipeline (SiT-B + moe1 Resume)

Notebook này là bản resume-only cho flow VAE + SiT-B + moe1 sau khi bạn giải nén lại output của notebook cũ vào `/kaggle/working`, nên chỉ còn phải dựng lại môi trường `uv` trong session Kaggle mới.

Artifact cần có sẵn sau bước giải nén output archive:

- `/kaggle/working/RAE`
- `/kaggle/working/celeba256_imgfolder`
- `/kaggle/working/celeba256_val_fid_stats_cpu.pkl`
- `/kaggle/working/celeba256_source_gmm.npz`
- `/kaggle/working/results_jax_tpu/` với ít nhất một thư mục run khớp mẫu `CelebA256_SiT-B_StabilityVAE_moe1_jax_tpuv5e8-*` và bên trong có `checkpoint_*`

Notebook này bắt đầu bằng cell giải nén `_output_.zip` của notebook cũ, sau đó mới kiểm tra repo, chạy `uv sync`, lấy Kaggle secret cho wandb, tự tìm run Orbax `SiT-B` mới nhất và checkpoint mới nhất bên trong nó, rồi resume train bằng `--workdir`.

Từ bản repo này trở đi, mỗi workdir JAX mới sẽ lưu thêm `wandb_run.json` để khóa đúng W&B run id của workdir. Khi resume, adapter sẽ tự rewind lịch sử W&B về latest `checkpoint_*` step rồi mới log tiếp, nên các đoạn metric đã ghi sau checkpoint cuối của session cũ sẽ không bị giữ lại sai lịch sử. Nếu bạn resume một legacy workdir cũ chưa có file này, hãy export `WANDB_RUN_ID=<existing_run_id>` trong session Kaggle trước khi chạy cell train; cell đó sẽ tự thêm `--wandb-run-id` đúng một lần để bind lại chính xác run W&B cũ trước khi tiếp tục.


In [ ]:
%%bash
set -euo pipefail

unzip -o /kaggle/input/notebooks/kieuhongquan/rae-jax/_output_.zip -d /kaggle/working 1>out.txt 2>err.txt


In [ ]:
%%bash
set -euo pipefail

[ -d /kaggle/working/RAE/.git ]
cd /kaggle/working/RAE
git rev-parse --short HEAD


In [ ]:
import os

os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/.venv"
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"

%cd /kaggle/working/RAE
!uv sync -q


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged


In [ ]:
from kaggle_secrets import UserSecretsClient
import os
import pathlib

user_secrets = UserSecretsClient()
wandb_token = user_secrets.get_secret("WANDB2")

os.environ["WANDB_API_KEY"] = wandb_token
os.environ["WANDB_KEY"] = wandb_token

netrc = pathlib.Path.home() / ".netrc"
netrc.write_text(f"machine api.wandb.ai login user password {wandb_token}\n")
os.chmod(netrc, 0o600)


In [ ]:
from pathlib import Path

repo_root = Path("/kaggle/working/RAE")
celeba_root = Path("/kaggle/working/celeba256_imgfolder")
source_gmm_path = Path("/kaggle/working/celeba256_source_gmm.npz")
fid_stats_path = Path("/kaggle/working/celeba256_val_fid_stats_cpu.pkl")
stage2_cfg_path = repo_root / "configs" / "stage2" / "training" / "CelebA256_SiT-B_StabilityVAE_moe1_jax_tpuv5e8.yaml"
results_root = Path("/kaggle/working/results_jax_tpu")
run_dirs = sorted(results_root.glob("CelebA256_SiT-B_StabilityVAE_moe1_jax_tpuv5e8-*"))
resume_workdir = run_dirs[-1] if run_dirs else None
checkpoint_dirs = sorted(resume_workdir.glob("checkpoint_*"), key=lambda path: int(path.name.split("_")[-1])) if resume_workdir else []
latest_ckpt_dir = checkpoint_dirs[-1] if checkpoint_dirs else None

for path in [repo_root, celeba_root, source_gmm_path, fid_stats_path, stage2_cfg_path, results_root]:
    print(path, "exists=", path.exists())
print("resume_workdir=", resume_workdir)
print("latest_ckpt_dir=", latest_ckpt_dir)

assert repo_root.exists()
assert celeba_root.exists()
assert source_gmm_path.exists()
assert fid_stats_path.exists()
assert stage2_cfg_path.exists()
assert results_root.exists()
assert resume_workdir is not None
assert latest_ckpt_dir is not None


## Resume Stage 2

Cell bên dưới tự tìm thư mục run `SiT-B + moe1` mới nhất trong `/kaggle/working/results_jax_tpu`, rồi dùng `--workdir` để resume trực tiếp từ Orbax run directory đó. Runtime sẽ tự restore `latest_step()` từ checkpoint mới nhất bên trong workdir; đoạn shell chỉ in ra run directory và thư mục `checkpoint_*` mới nhất để bạn kiểm tra nhanh trước khi train.


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

results_root="/kaggle/working/results_jax_tpu"
resume_workdir=$(find "${results_root}" -maxdepth 1 -mindepth 1 -type d -name "CelebA256_SiT-B_StabilityVAE_moe1_jax_tpuv5e8-*" | sort -V | tail -n 1)
[ -n "${resume_workdir}" ]
latest_ckpt=$(find "${resume_workdir}" -maxdepth 1 -type d -name "checkpoint_*" | sort -V | tail -n 1)
wandb_args=()
if [ ! -f "${resume_workdir}/wandb_run.json" ]; then
  if [ -n "${WANDB_RUN_ID:-}" ]; then
    wandb_args+=(--wandb-run-id "${WANDB_RUN_ID}")
  else
    echo "legacy workdir without wandb_run.json; set WANDB_RUN_ID to the historical W&B run id before resuming" >&2
  fi
fi
[ -n "${latest_ckpt}" ]
resume_exp_name=$(basename "${resume_workdir}")
echo "resume workdir: ${resume_workdir}"
echo "latest checkpoint: ${latest_ckpt}"
echo "resume exp_name: ${resume_exp_name}"

export ENTITY="TungBangDSLab"
export PROJECT="moe-diffusion"

uv run python src_jax/train.py \
  --config configs/stage2/training/CelebA256_SiT-B_StabilityVAE_moe1_jax_tpuv5e8.yaml \
  --data-path /kaggle/working/celeba256_imgfolder \
  --workdir "${resume_workdir}" \
  --exp-name "${resume_exp_name}" \
  --precision bf16 \
  --wandb \
  --wandb-entity TungBangDSLab \
  --wandb-project "${PROJECT}" \
  "${wandb_args[@]}" \
  --set training.global_batch_size=64 \
  --set training.num_workers=16 \
  --set training.prefetch_factor=4 \
  --set training.log_rae_latent_stats=true \
  --set training.log_activation_stats=true \
  --set training.ckpt_every=290000 \
  --set eval.num_workers=16 \
  --set eval.prefetch_factor=4 \
  --set eval.fid_ref=/kaggle/working/celeba256_val_fid_stats_cpu.pkl \
  --set eval.fid_every=10000 \
  --set eval.fid_num_samples=4096
